In [14]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [15]:
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.datasets import cifar10, mnist
from sklearn.manifold import TSNE

Loading dataset

In [16]:
def load_dataset(dataset="cifar10"):
    if dataset == "cifar10":
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
        input_shape = (32, 32, 3)
    else:
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
        x_train = np.expand_dims(x_train, -1)
        x_test = np.expand_dims(x_test, -1)
        input_shape = (28, 28, 1)

    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0

    y_train = tf.keras.utils.to_categorical(y_train, 10)
    y_test = tf.keras.utils.to_categorical(y_test, 10)

    return x_train, y_train, x_test, y_test, input_shape

LeNet-5

In [23]:
def LeNet(input_shape):
    model = models.Sequential([
        layers.Conv2D(6, kernel_size=5, activation='relu', input_shape=input_shape),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Conv2D(16, kernel_size=5, activation='relu'),
        layers.AveragePooling2D(pool_size=(2,2)),
        layers.Flatten(),
        layers.Dense(120, activation='relu'),
        layers.Dense(84, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

AlexNet

In [18]:
def AlexNet(input_shape):
    model = models.Sequential([
        layers.Conv2D(96, 3, activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(),
        layers.Conv2D(256, 3, activation='relu'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(1024, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

VGG16

In [19]:
def VGG():
    base = tf.keras.applications.VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(32, 32, 3)
    )
    base.trainable = False

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ])
    return model

RestNet50

In [20]:
def ResNet():
    base = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(32, 32, 3)
    )
    base.trainable = False

    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(10, activation='softmax')
    ])
    return model

In [21]:
def train_model(model, x_train, y_train, x_test, y_test,
                loss, optimizer, epochs):
    
    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=["accuracy"]
    )

    history = model.fit(
        x_train, y_train,
        epochs=epochs,
        batch_size=64,
        validation_split=0.1,
        verbose=1
    )

    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    return history, test_acc

# part 1 : CNN comparisions for cifar-10

In [24]:
x_train, y_train, x_test, y_test, input_shape = load_dataset("cifar10")

models_dict = {
    "LeNet": LeNet(input_shape),
    "AlexNet": AlexNet(input_shape),
    "VGG": VGG(),
    "ResNet": ResNet()
}

results = {}

for name, model in models_dict.items():
    print("\nTraining:", name)
    _, acc = train_model(
        model,
        x_train, y_train,
        x_test, y_test,
        loss="categorical_crossentropy",
        optimizer="adam",
        epochs=5
    )
    results[name] = acc

results

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 118s 2us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 301s 3us/step

Training: LeNet
Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.3913 - loss: 1.6711 - val_accuracy: 0.4688 - val_loss: 1.4681
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4905 - loss: 1.4208 - val_accuracy: 0.5226 - val_loss: 1.3445
Epoch 3/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5337 - loss: 1.3136 - val_accuracy: 0.5422 - val_loss: 1.2901
Epoch 4/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5634 - loss: 1.2315 - val_accuracy: 0.5746 - val_loss: 1.2121
Epoch 5/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.5881 - loss: 1.1681 - val_accuracy: 0.5996 - val_loss: 1.1536

Training: AlexNet
Epoch 1/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 1069s 2s/step - accuracy: 0.5082 - loss: 1.3743 - val_accuracy: 0.6130 - val_loss: 1.0944
Epoch 2/5
704/704 ━━━━━━━━━━━━━━━━━━━━ 6912s 10s/step - accuracy: 0.6506 - loss: 1.0012 - val_

{'LeNet': 0.5792999863624573,
 'AlexNet': 0.7281000018119812,
 'VGG': 0.5623999834060669,
 'ResNet': 0.3352000117301941}